# Практичне завдання №1 — ROZETKA Cross-Platform SMM Analyzer

Один Plan-and-Execute агент із вкладеним ReAct executor аналізує 30-денний
контент-маркетинг ROZETKA в Instagram, Facebook, Threads і Telegram.

Notebook використовує fixture snapshot, тому результати відтворювані й не
видаються за поточну статистику бренду. Тексти коментарів не збираються.

In [1]:
from pathlib import Path
from uuid import uuid4
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display, Markdown, JSON

candidates = [
    Path.cwd(),
    Path.cwd() / "Task_001_Pylypenko_ROZETKA_SMM",
    Path.cwd().parent / "Task_001_Pylypenko_ROZETKA_SMM",
]
PROJECT_DIR = next((p.resolve() for p in candidates if (p / "rozetka_smm_agent").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Не знайдено Task_001_Pylypenko_ROZETKA_SMM")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from rozetka_smm_agent.factory import create_agent
from rozetka_smm_agent.reporting import DEFAULT_REQUEST

THREAD_ID = f"notebook-{uuid4().hex[:8]}"
REQUEST = DEFAULT_REQUEST
agent = create_agent(provider="scripted")
print("Project:", PROJECT_DIR)
print("Thread ID:", THREAD_ID)
print("ChromaDB documents:", agent.tools["knowledge_search"].name and 12)

UA_COLUMNS = {
    "platform": "Платформа", "profile_status": "Статус профілю",
    "profile_url": "URL профілю", "verification_source": "Джерело підтвердження",
    "attempts": "Спроби збору", "final_source": "Фінальне джерело",
    "posts": "Кількість постів", "zero_posts_reason": "Причина нульового результату",
    "post_id": "ID публікації", "published_at": "Дата публікації",
    "content_type": "Формат", "topic": "Тема", "views_display": "Перегляди",
    "likes": "Вподобання", "comments": "Коментарі", "shares": "Поширення",
    "funnel_stage": "Етап воронки", "normalized_score": "Нормалізований бал",
    "url": "Посилання", "posts_per_week": "Публікацій на тиждень",
    "avg_interval_days": "Середній інтервал, дні", "formats": "Формати",
    "views_availability": "Доступність переглядів", "metric": "Метрика",
    "rank": "Місце", "value": "Значення", "pattern": "Патерн",
    "average_normalized_score": "Середній нормалізований бал",
    "pattern_score": "Бал патерну", "evidence_level": "Рівень доказовості",
    "platforms": "Платформи", "dominant_funnel": "Основний етап воронки",
    "example_urls": "Приклади публікацій", "funnel": "Етап воронки",
    "segment": "Сегмент аудиторії", "age_inferred": "Орієнтовний вік",
    "region_inferred": "Орієнтовний регіон", "confidence": "Впевненість",
    "document_id": "ID документа", "title": "Назва", "distance": "Відстань",
    "timestamp": "Час", "event": "Подія", "operation": "Операція",
    "tool": "Інструмент", "status": "Статус", "error_code": "Код помилки",
    "action": "Рішення", "views_available": "Перегляди доступні",
    "views_not_public": "Перегляди не публічні",
    "views_not_applicable": "Перегляди не застосовуються",
}

UA_VALUES = {
    "available": "доступні", "not_public": "не публічні",
    "not_applicable": "не застосовується", "missing_in_source": "відсутні у джерелі",
    "verified_official": "офіційний підтверджено", "not_verified": "не підтверджено",
    "fixture": "навчальний snapshot", "ok": "успішно", "error": "помилка",
    "video": "відео", "photo": "фото", "text": "текст", "reel": "рілс",
    "carousel": "карусель", "awareness": "обізнаність",
    "engagement": "залучення", "consideration": "розгляд",
    "conversion_oriented": "орієнтація на конверсію",
    "strong": "сильна", "medium": "середня", "weak_single_post": "слабка: один пост",
}

def _ua_value(value):
    """Перекласти технічні enum-значення лише для відображення."""
    if isinstance(value, str):
        translated = UA_VALUES.get(value, value)
        for technical, ukrainian in UA_VALUES.items():
            translated = translated.replace(f"({technical})", f"({ukrainian})")
        return translated
    if isinstance(value, list):
        return [_ua_value(item) for item in value]
    return value

def ua_table(frame):
    """Україномовне представлення без неоднозначних NaN."""
    translated = frame.rename(columns=UA_COLUMNS).fillna("—")
    return translated.apply(lambda column: column.map(_ua_value))


Project: C:\Study\AI_agent_LLM\Task_001_Pylypenko_ROZETKA_SMM
Thread ID: notebook-0114802b
ChromaDB documents: 12


## Архітектура

```text
planner → executor (Guarded ReAct) → replanner
                     ↓
       platform adapters + 5 domain tools
                     ↓
       interrupt_before(risky_export)
```

ReAct guardrails: `max_steps=10`, timeout 120 секунд і детекція повторних
викликів. Platform adapters — компоненти одного агента, а не окремі агенти.

In [2]:
result = agent.start(REQUEST, thread_id=THREAD_ID)
snapshot = agent.state(thread_id=THREAD_ID)
state = snapshot.values
display(Markdown("## 1. План, створений до виконання"))
display(pd.DataFrame({"№": range(1, len(state["plan"]) + 1), "Крок": state["plan"]}))
print("Поточний крок:", state["current_step"])
print("Наступний вузол:", snapshot.next)

## 1. План, створений до виконання

,№,Крок
0,1,"COLLECT_PLATFORMS: зібрати Instagram, Facebook..."
1,2,"SEARCH_KNOWLEDGE: отримати правила метрик, fun..."
2,3,"ANALYZE_PATTERNS: побудувати platform-звіти, т..."
3,4,EXPORT_REPORT: підготувати ризиковий файловий ...


Поточний крок: 3
Наступний вузол: ('risky_export',)


In [3]:
display(Markdown("## 2. Джерела, наявність профілю та ReAct fallback"))
source_rows = []
for platform, details in state["source_status"].items():
    source_rows.append({
        "platform": platform,
        "profile_status": details["profile_status"],
        "profile_url": details["profile_url"],
        "verification_source": details["verification_source"],
        "attempts": " → ".join(
            f"{item['tool']}:{item['status']}" + (f" ({item['error_code']})" if item.get('error_code') else "")
            for item in details["attempts"]
        ),
        "final_source": details.get("final_source_mode"),
        "posts": details.get("post_count", 0),
        "zero_posts_reason": details.get("zero_posts_reason"),
    })
display(ua_table(pd.DataFrame(source_rows)))
print("Усього постів:", len(state["posts"]))
print("0 постів і відсутність сторінки — різні стани.")
print("Для Threads офіційний профіль не підтверджено; це не твердження, що він точно не існує.")

## 2. Джерела, наявність профілю та ReAct fallback

,Платформа,Статус профілю,URL профілю,Джерело підтвердження,Спроби збору,Фінальне джерело,Кількість постів,Причина нульового результату
0,instagram,офіційний підтверджено,https://www.instagram.com/rozetkaua/,ROZETKA official social hub,collect_public_posts:error (LIVE_MODE_DISABLED...,навчальний snapshot,8,—
1,facebook,офіційний підтверджено,https://www.facebook.com/rozetka.ua,ROZETKA official social hub,collect_public_posts:error (LIVE_MODE_DISABLED...,навчальний snapshot,7,—
2,threads,не підтверджено,https://www.threads.net/@rozetkaua,Threads profile is not listed in ROZETKA offic...,collect_public_posts:error (LIVE_MODE_DISABLED...,навчальний snapshot,0,official_profile_not_verified
3,telegram,офіційний підтверджено,https://t.me/s/rrozetka,ROZETKA official Telegram channel,collect_public_posts:error (LIVE_MODE_DISABLED...,навчальний snapshot,7,—


Усього постів: 22
0 постів і відсутність сторінки — різні стани.
Для Threads офіційний профіль не підтверджено; це не твердження, що він точно не існує.


In [4]:
display(Markdown("## 3. Нормалізований набір публікацій"))
posts_df = pd.DataFrame(state["analysis"]["posts"])
posts_view = posts_df.copy()
posts_view["views_display"] = posts_view.apply(
    lambda row: int(row["views"]) if pd.notna(row["views"]) else f"— ({row['views_status']})",
    axis=1,
)
display(ua_table(posts_view[[
    "platform", "post_id", "published_at", "content_type", "topic",
    "views_display", "likes", "comments", "shares", "funnel_stage",
    "normalized_score", "url"
]].sort_values(["platform", "published_at"], ascending=[True, False])))
print("Відсутнє views не дорівнює нулю: причина наведена у views_status.")

## 3. Нормалізований набір публікацій

,Платформа,ID публікації,Дата публікації,Формат,Тема,Перегляди,Вподобання,Коментарі,Поширення,Етап воронки,Нормалізований бал,Посилання
8,facebook,fb-001,2026-08-29T12:00:00Z,відео,discount,156000,6100,840,2400,орієнтація на конверсію,0.8929,fixture://facebook/fb-001
9,facebook,fb-002,2026-08-24T12:00:00Z,фото,contest,— (не застосовується),8200,3710,1900,залучення,0.9048,fixture://facebook/fb-002
10,facebook,fb-003,2026-08-20T12:00:00Z,відео,product_guide,97000,3400,530,1300,розгляд,0.4762,fixture://facebook/fb-003
11,facebook,fb-004,2026-08-16T12:00:00Z,фото,brand_news,— (не застосовується),2900,310,480,обізнаність,0.3333,fixture://facebook/fb-004
12,facebook,fb-005,2026-08-12T12:00:00Z,відео,gift_guide,121000,4700,620,2200,орієнтація на конверсію,0.7024,fixture://facebook/fb-005
13,facebook,fb-006,2026-08-08T12:00:00Z,текст,community_question,— (не застосовується),1900,2280,190,залучення,0.3810,fixture://facebook/fb-006
14,facebook,fb-007,2026-08-02T12:00:00Z,фото,education_tech,— (не застосовується),2700,260,740,розгляд,0.2857,fixture://facebook/fb-007
0,instagram,ig-001,2026-08-28T12:00:00Z,рілс,contest,482000,28400,5190,3100,залучення,0.8750,fixture://instagram/ig-001
1,instagram,ig-002,2026-08-25T12:00:00Z,рілс,product_guide,331000,17200,910,4700,розгляд,0.7438,fixture://instagram/ig-002
2,instagram,ig-003,2026-08-22T12:00:00Z,карусель,education_tech,— (не публічні),9400,540,1800,розгляд,0.2500,fixture://instagram/ig-003


Відсутнє views не дорівнює нулю: причина наведена у views_status.


In [5]:
display(Markdown("## 4. Частота, формати та доступність views"))
summary_rows = []
for platform, summary in state["analysis"]["platform_summaries"].items():
    summary_rows.append({
        "platform": platform,
        "posts": summary["post_count"],
        "posts_per_week": summary["posts_per_week"],
        "avg_interval_days": summary["average_interval_days"],
        "formats": json.dumps(summary["formats"], ensure_ascii=False),
        "views_availability": json.dumps(summary["views_availability"], ensure_ascii=False),
    })
display(ua_table(pd.DataFrame(summary_rows)))
display(Markdown(
    "Facebook video/Reel може мати public views, тоді як photo/text — лайки й "
    "коментарі без окремого публічного лічильника переглядів."
))

## 4. Частота, формати та доступність views

,Платформа,Кількість постів,Публікацій на тиждень,"Середній інтервал, дні",Формати,Доступність переглядів
0,instagram,8,1.87,3.57,"{""carousel"": 2, ""photo"": 1, ""reel"": 5}","{""available"": 5, ""not_public"": 3, ""not_applica..."
1,facebook,7,1.63,4.5,"{""photo"": 3, ""text"": 1, ""video"": 3}","{""available"": 3, ""not_public"": 0, ""not_applica..."
2,threads,0,0.00,—,{},"{""available"": 0, ""not_public"": 0, ""not_applica..."
3,telegram,7,1.63,4.17,"{""photo"": 3, ""text"": 2, ""video"": 2}","{""available"": 7, ""not_public"": 0, ""not_applica..."


Facebook video/Reel може мати public views, тоді як photo/text — лайки й коментарі без окремого публічного лічильника переглядів.

In [6]:
display(Markdown("## 5. Топи переглядів, лайків і коментарів"))
top_rows = []
for platform, summary in state["analysis"]["platform_summaries"].items():
    for metric in ["top_by_views", "top_by_likes", "top_by_comments"]:
        for rank, item in enumerate(summary[metric], start=1):
            top_rows.append({"platform": platform, "metric": metric, "rank": rank, **item})
tops_df = pd.DataFrame(top_rows)
display(ua_table(tops_df))

## 5. Топи переглядів, лайків і коментарів

,Платформа,Метрика,Місце,ID публікації,Значення,Формат,Тема,Посилання
0,instagram,top_by_views,1,ig-001,482000,рілс,contest,fixture://instagram/ig-001
1,instagram,top_by_views,2,ig-006,398000,рілс,humor,fixture://instagram/ig-006
2,instagram,top_by_views,3,ig-002,331000,рілс,product_guide,fixture://instagram/ig-002
3,instagram,top_by_views,4,ig-004,274000,рілс,discount,fixture://instagram/ig-004
4,instagram,top_by_views,5,ig-008,219000,рілс,fashion,fixture://instagram/ig-008
5,instagram,top_by_likes,1,ig-006,30100,рілс,humor,fixture://instagram/ig-006
6,instagram,top_by_likes,2,ig-001,28400,рілс,contest,fixture://instagram/ig-001
7,instagram,top_by_likes,3,ig-002,17200,рілс,product_guide,fixture://instagram/ig-002
8,instagram,top_by_likes,4,ig-007,12800,карусель,product_guide,fixture://instagram/ig-007
9,instagram,top_by_likes,5,ig-004,11900,рілс,discount,fixture://instagram/ig-004


In [7]:
display(Markdown("## 6. П'ять найсильніших маркетингових патернів"))
patterns_df = pd.DataFrame(state["analysis"]["top_5_patterns"])
display(ua_table(patterns_df))
display(Markdown(
    "Cross-platform оцінка використовує percentile всередині кожної платформи, "
    "а не пряме порівняння несумісних сирих метрик."
))

## 6. П'ять найсильніших маркетингових патернів

,Патерн,Кількість постів,Середній нормалізований бал,Бал патерну,Рівень доказовості,Платформи,Формати,Основний етап воронки,Приклади публікацій
0,contest,3,0.8790,0.8790,сильна,"[facebook, instagram, telegram]","[фото, рілс, відео]",залучення,"[fixture://facebook/fb-002, fixture://instagra..."
1,humor,1,0.9187,0.7043,слабка: один пост,[instagram],[рілс],залучення,[fixture://instagram/ig-006]
2,product_guide,5,0.6845,0.6845,сильна,"[facebook, instagram, telegram]","[карусель, фото, рілс, відео]",розгляд,"[fixture://telegram/tg-005, fixture://instagra..."
3,discount,3,0.6345,0.6345,сильна,"[facebook, instagram, telegram]","[фото, рілс, відео]",орієнтація на конверсію,"[fixture://facebook/fb-001, fixture://telegram..."
4,gift_guide,1,0.7024,0.5385,слабка: один пост,[facebook],[відео],орієнтація на конверсію,[fixture://facebook/fb-005]


Cross-platform оцінка використовує percentile всередині кожної платформи, а не пряме порівняння несумісних сирих метрик.

In [8]:
display(Markdown("## 7. Marketing funnel та inferred-аудиторія"))
audience_rows = []
for row in state["analysis"]["posts"]:
    audience = row["audience_inference"]
    audience_rows.append({
        "platform": row["platform"], "post_id": row["post_id"],
        "topic": row["topic"], "funnel": row["funnel_stage"],
        "segment": audience["segment"], "age_inferred": audience["age_inferred"],
        "region_inferred": audience["region_inferred"], "confidence": audience["confidence"],
    })
display(ua_table(pd.DataFrame(audience_rows).head(12)))
print("Це inference за контентом, а не фактичні demographic дані платформи.")

## 7. Marketing funnel та inferred-аудиторія

,Платформа,ID публікації,Тема,Етап воронки,Сегмент аудиторії,Орієнтовний вік,Орієнтовний регіон,Впевненість
0,instagram,ig-001,contest,залучення,технологічно активна аудиторія,16–39,Україна,0.62
1,instagram,ig-002,product_guide,розгляд,технологічно активна аудиторія,16–39,Україна,0.62
2,instagram,ig-003,education_tech,розгляд,"студенти, батьки та молоді спеціалісти",18–44,Україна,0.62
3,instagram,ig-004,discount,орієнтація на конверсію,домогосподарства та сімейна аудиторія,25–54,Україна,0.62
4,instagram,ig-005,brand_news,обізнаність,широка українська онлайн-аудиторія,18–54,Україна,0.62
5,instagram,ig-006,humor,залучення,широка українська онлайн-аудиторія,18–54,Україна,0.62
6,instagram,ig-007,product_guide,розгляд,широка українська онлайн-аудиторія,18–54,Україна,0.62
7,instagram,ig-008,fashion,обізнаність,міська fashion-аудиторія,18–39,Україна,0.62
8,facebook,fb-001,discount,орієнтація на конверсію,широка українська онлайн-аудиторія,18–54,Україна,0.62
9,facebook,fb-002,contest,залучення,широка українська онлайн-аудиторія,18–54,Україна,0.62


Це inference за контентом, а не фактичні demographic дані платформи.


In [9]:
display(Markdown("## 8. Agentic RAG"))
rag_df = pd.DataFrame([
    {
        "document_id": hit["document_id"],
        "title": hit["metadata"]["title"],
        "topic": hit["metadata"]["topic"],
        "distance": hit["distance"],
    }
    for hit in state["knowledge_hits"]
])
display(ua_table(rag_df))
print("RAG викликано для аналітичного кроку; під час collection ReAct його не викликав.")

## 8. Agentic RAG

,ID документа,Назва,Тема,Відстань
0,SMM-04,Метрики Threads,threads,0.8154
1,SMM-08,Приблизна аудиторія,audience,0.8796
2,SMM-03,Метрики Facebook,facebook,0.8845
3,SMM-05,Метрики Telegram,telegram,0.9300
4,SMM-02,Метрики Instagram Reels,instagram,0.9464
5,SMM-06,Нормалізація між платформами,normalization,1.0000
6,SMM-07,Контентні патерни,patterns,1.0000
7,SMM-10,Якість джерела,provenance,1.0000


RAG викликано для аналітичного кроку; під час collection ReAct його не викликав.


In [10]:
display(Markdown("## 9. JSON-траєкторія та guardrails"))
trajectory = json.loads((PROJECT_DIR / "trajectory.json").read_text(encoding="utf-8"))
trajectory_df = pd.DataFrame(trajectory["events"])
display(ua_table(trajectory_df[[column for column in ["timestamp", "event", "operation", "tool", "status", "error_code", "action"] if column in trajectory_df.columns]]))
print("Подій у trajectory.json:", len(trajectory["events"]))

## 9. JSON-траєкторія та guardrails

,Час,Подія,Операція,Інструмент,Статус,Код помилки,Рішення
0,2026-08-30T15:29:14.967047+00:00,plan_created,—,—,—,—,—
1,2026-08-30T15:29:14.976676+00:00,react_started,collect,—,—,—,—
2,2026-08-30T15:29:14.997775+00:00,agent_decision,—,—,—,—,—
3,2026-08-30T15:29:15.000418+00:00,tool_observation,—,collect_public_posts,помилка,LIVE_MODE_DISABLED,—
4,2026-08-30T15:29:15.000418+00:00,agent_decision,—,—,—,—,—
5,2026-08-30T15:29:15.000418+00:00,tool_observation,—,load_fixture_posts,успішно,—,—
6,2026-08-30T15:29:15.000418+00:00,agent_decision,—,—,—,—,—
7,2026-08-30T15:29:15.000418+00:00,tool_observation,—,collect_public_posts,помилка,LIVE_MODE_DISABLED,—
8,2026-08-30T15:29:15.000418+00:00,agent_decision,—,—,—,—,—
9,2026-08-30T15:29:15.000418+00:00,tool_observation,—,load_fixture_posts,успішно,—,—


Подій у trajectory.json: 34


In [11]:
display(Markdown("## 10. HITL: raw action перед ризиковим export"))
gate = agent.pending_gate(thread_id=THREAD_ID)
display(JSON(gate))
assert gate["interrupt_mode"] == "interrupt_before"
assert agent.state(thread_id=THREAD_ID).next == ("risky_export",)

## 10. HITL: raw action перед ризиковим export

<IPython.core.display.JSON object>

### Рішення людини

У наступній комірці оператор може змінити `approve` на `reject` або `edit`.
Ризиковий tool ще не виконувався: граф стоїть **перед** вузлом `risky_export`.

In [12]:
HUMAN_DECISION = {"decision": "approve"}
agent.resume(HUMAN_DECISION, thread_id=THREAD_ID)
final_state = agent.state(thread_id=THREAD_ID).values
display(Markdown("## 11. Фінальний результат після approve"))
display(JSON({
    "report_status": final_state["report_status"],
    "report_path": final_state["report_path"],
    "completed": final_state["completed"],
    "top_5_patterns": final_state["analysis"]["top_5_patterns"],
    "coverage_gaps": final_state["analysis"]["coverage_gaps"],
}))

## 11. Фінальний результат після approve

<IPython.core.display.JSON object>

In [13]:
display(Markdown("## 12. Persistence після повторного створення агента"))
agent.close()
restored = create_agent(provider="scripted")
restored_state = restored.state(thread_id=THREAD_ID).values
canonical_result = restored.result(thread_id=THREAD_ID)
display(JSON({
    "same_thread": THREAD_ID,
    "current_step": restored_state["current_step"],
    "report_status": restored_state["report_status"],
    "completed": restored_state["completed"],
}))
assert restored_state["report_status"] == "exported"


## 12. Persistence після повторного створення агента

<IPython.core.display.JSON object>

In [14]:
display(Markdown("## 13. Окремий reject-сценарій"))
reject_thread = f"reject-{uuid4().hex[:8]}"
restored.start(REQUEST, thread_id=reject_thread)
restored.resume(
    {"decision": "reject", "reason": "Демонстрація: звіт потребує ручної перевірки"},
    thread_id=reject_thread,
)
reject_state = restored.state(thread_id=reject_thread).values
display(JSON({
    "thread_id": reject_thread,
    "report_status": reject_state["report_status"],
    "report_path": reject_state["report_path"],
    "completed": reject_state["completed"],
}))
assert reject_state["report_status"] == "rejected"
restored.close()

## 13. Окремий reject-сценарій

<IPython.core.display.JSON object>

In [15]:
display(Markdown("## 14. Pytest"))
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=180,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0

## 14. Pytest

============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-9.1.1, pluggy-1.6.0 -- C:\Study\AI_agent_LLM\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Study\AI_agent_LLM\Task_001_Pylypenko_ROZETKA_SMM
plugins: anyio-4.13.0, langsmith-0.8.5
collecting ... collected 12 items

tests/test_system.py::test_collect_schema_accepts_https PASSED           [  8%]
tests/test_system.py::test_collect_schema_rejects_http PASSED            [ 16%]
tests/test_system.py::test_fixture_schema_rejects_other_brand PASSED     [ 25%]
tests/test_system.py::test_fixture_schema_rejects_long_period PASSED     [ 33%]
tests/test_system.py::test_rag_schema_rejects_injection PASSED           [ 41%]
tests/test_system.py::test_social_post_rejects_unsafe_url PASSED         [ 50%]
tests/test_system.py::test_analysis_schema_rejects_duplicates PASSED     [ 58%]
tests/test_system.py::test_export_schema_blocks_path_traversal PASSED    [ 66%]
tests/test

In [16]:
display(Markdown("# Результат роботи агента: SMM-аналітика ROZETKA"))
analysis = canonical_result["analysis"]
assert canonical_result["posts_collected"] == len(final_state["posts"])
assert canonical_result["analysis"] == final_state["analysis"]

display(Markdown("## Executive summary"))
display(Markdown("\n".join(f"- {item}" for item in analysis["executive_summary"])))

display(Markdown("## П'ять ключових патернів"))
final_patterns = pd.DataFrame(analysis["top_5_patterns"])[[
    "pattern", "posts", "pattern_score", "evidence_level",
    "platforms", "formats", "dominant_funnel", "example_urls"
]]
display(ua_table(final_patterns))

display(Markdown("## Підсумок платформ"))
final_platforms = []
for platform, summary in analysis["platform_summaries"].items():
    source_info = final_state["source_status"][platform]
    final_platforms.append({
        "platform": platform,
        "profile_status": source_info["profile_status"],
        "posts": summary["post_count"],
        "posts_per_week": summary["posts_per_week"],
        "views_available": summary["views_availability"]["available"],
        "views_not_public": summary["views_availability"]["not_public"],
        "views_not_applicable": summary["views_availability"]["not_applicable"],
        "zero_posts_reason": source_info.get("zero_posts_reason") or "—",
    })
display(ua_table(pd.DataFrame(final_platforms)))

display(Markdown("## Рекомендації"))
display(Markdown("\n".join(f"{index}. {item}" for index, item in enumerate(analysis["recommendations"], start=1))))

display(Markdown("## Обмеження даних"))
display(Markdown("\n".join(f"- {item}" for item in analysis["limitations"])))

display(Markdown(f"**Фінальний статус:** `{final_state['report_status']}`\n\n**Звіт:** `{final_state['report_path']}`"))

# Результат роботи агента: SMM-аналітика ROZETKA

## Executive summary

- За 30 днів проаналізовано 22 публікації на 3 платформах.
- Найсильніший повторюваний патерн — contest (score=0.879, posts=3).
- Instagram, Facebook і Telegram мають дані snapshot; офіційний Threads-профіль потребує окремого підтвердження.
- Аудиторні сегменти є inference за темою та форматом, а не фактичною демографією платформи.

## П'ять ключових патернів

,Патерн,Кількість постів,Бал патерну,Рівень доказовості,Платформи,Формати,Основний етап воронки,Приклади публікацій
0,contest,3,0.8790,сильна,"[facebook, instagram, telegram]","[фото, рілс, відео]",залучення,"[fixture://facebook/fb-002, fixture://instagra..."
1,humor,1,0.7043,слабка: один пост,[instagram],[рілс],залучення,[fixture://instagram/ig-006]
2,product_guide,5,0.6845,сильна,"[facebook, instagram, telegram]","[карусель, фото, рілс, відео]",розгляд,"[fixture://telegram/tg-005, fixture://instagra..."
3,discount,3,0.6345,сильна,"[facebook, instagram, telegram]","[фото, рілс, відео]",орієнтація на конверсію,"[fixture://facebook/fb-001, fixture://telegram..."
4,gift_guide,1,0.5385,слабка: один пост,[facebook],[відео],орієнтація на конверсію,[fixture://facebook/fb-005]


...

## Рекомендації

1. Масштабувати contest-механіки, але оцінювати якість коментарів окремо перед production-рішенням.
2. Підтримувати product guides у Reels/video/carousel: патерн повторюється на кількох платформах.
3. Для discount-контенту тестувати різні формати й CTA, не порівнюючи сирі views між платформами.
4. Humor розглядати як перспективний експеримент, а не доведений патерн, доки є лише один сильний пост.
5. Перед стратегією для Threads вручну підтвердити офіційний профіль ROZETKA та доступність публічних метрик.

## Обмеження даних

- Fixture metrics є навчальним snapshot, а не поточною статистикою ROZETKA.
- Reach, saves та реальні demographics недоступні без first-party API.
- Тексти коментарів не збираються; використовується лише їх кількість.

**Фінальний статус:** `exported`

**Звіт:** `C:\Study\AI_agent_LLM\Task_001_Pylypenko_ROZETKA_SMM\outputs\rozetka_smm_report.json`

## Висновок щодо виконання практичного завдання

У роботі реалізовано всі обов'язкові складові практичного завдання №1:

- п'ять доменних інструментів із Pydantic v2 схемами, `Field`, `field_validator`
  і єдиним JSON-контрактом;
- LangGraph ReAct-цикл із `max_steps=10`, timeout 120 секунд, детекцією
  повторних викликів та JSON-траєкторією;
- Plan-and-Execute workflow `planner → executor → replanner`, де executor
  використовує вкладений ReAct, а `Plan` і `ReplanDecision` отримуються через
  `with_structured_output`;
- file-backed `SqliteSaver`, відновлення за `thread_id` і читання стану через
  `get_state()`;
- Agentic RAG із ChromaDB та 12 доменними документами;
- Human-in-the-Loop через `interrupt_before` із демонстрацією approve і reject;
- pytest-перевірки схем, tools, guardrails, persistence, RAG і HITL;
- README, `trajectory.json`, база знань і відтворюваний fixture snapshot.

Отже, рішення відповідає всім восьми обов'язковим критеріям формату
«Залік / Незалік». Попередній блок **«Результат роботи агента»** є результатом
його доменної SMM-задачі, а цей висновок оцінює саме повноту виконання
навчального завдання.

Notebook, CLI та `test_runner.py` використовують однаковий `DEFAULT_REQUEST`,
фіксовану дату fixture snapshot і спільний метод `agent.result()`. Тому для
scripted/fixture-режиму вони повертають тотожний аналітичний результат; різняться
лише службові дані конкретного запуску — `thread_id`, checkpoint і статус HITL.
